# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

**Method, step by step:**

1. **Finds the data.** In Google Colab, the notebook starts on an empty machine, so it downloads a copy of the project (`git clone --depth 1` fetches only the latest version, which is faster). Run locally, it starts inside `work/notebooks/`, so it steps up one folder at a time until it reaches the project's top folder, where `data/raw/` lives.
2. **Loads the tools.** `pandas` handles tables and `scipy`'s `poisson` is the probability model used later to test whether a page's missing clicks could be bad luck.
3. **Loads the dataset** into a table called `df`, stopping with a clear message if the file is missing.
4. **Defines `show()` and `show_table()`**, two small helpers that display every result as a titled table in the same layout, so each cell's output is easy to scan.


In [ ]:
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import poisson

# Colab starts empty, so clone the project there; locally, walk up to the folder that holds data/raw.
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/neilivanorencia/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

DATA = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA), "Starter CSV not found: run this notebook from inside the project folder."
df = pd.read_csv(DATA)
pd.set_option("display.width", 160)
TIERS = ["top_3", "page_1", "striking", "page_3_5", "deep"]


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Dataset loaded", {
    "pages (rows)": f"{len(df):,}",
    "columns": df.shape[1],
    "clients": df["client_id"].nunique(),
})

### Shared definitions

**Method:** this cell defines four small functions used throughout this notebook, then builds the lane's table of pages.

- `tier_expected()` gives each page its expected clicks from its tier's pooled CTR (as in Week 1).
- `client_aware_expected()` adjusts that expectation for the page's client, so a client whose pages always click less is not mistaken for a site full of problems.
- `precision_at_k()` and `capture_at_k()` compute the two success measures defined above.

It then keeps pages with real position data and impressions in both windows, and computes expected clicks, p, and missed clicks separately for window A and window B.


In [ ]:
# Window A is the earlier 30 days and window B the most recent 30 days; they never overlap.
def tier_expected(frame: pd.DataFrame, clicks: str, impressions: str) -> pd.Series:
    """Return each page's expected clicks from the pooled CTR of its position tier.

    Pooling clicks and impressions per tier, rather than averaging page CTRs, keeps small pages
    from dragging the expectation around.
    """
    rate = frame.groupby("position_tier")[clicks].sum() / frame.groupby("position_tier")[impressions].sum()
    return frame[impressions] * frame["position_tier"].map(rate)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str) -> pd.Series:
    """Return the tier expectation scaled by how each client's pages click relative to that norm.

    The +1 in the numerator and denominator keeps a client with very few clicks from receiving
    an extreme factor.
    """
    expected = tier_expected(frame, clicks, impressions)
    by_client = frame.assign(_expected=expected).groupby("client_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_id"].map(factor)


def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the share of the top-k pages by score whose target is 1."""
    top = score.sort_values(ascending=False).index[:k]
    return float(target.loc[top].mean())


def capture_at_k(score: pd.Series, value: pd.Series, k: int) -> float:
    """Return the share of the total value (for example, missed clicks) that sits in the top-k pages."""
    top = score.sort_values(ascending=False).index[:k]
    return float(value.loc[top].sum() / value.sum())


# The lane frame: pages with real position data and impressions in both 30-day windows.
lane = df[(df["avg_position"] > 0) & (df["impressions_prev_30d"] > 0) & (df["impressions_last_30d"] > 0)].copy()
for tag, clicks, impressions in [("A", "clicks_prev_30d", "impressions_prev_30d"),
                                 ("B", "clicks_last_30d", "impressions_last_30d")]:
    lane[f"expected_{tag}"] = client_aware_expected(lane, clicks, impressions)
    lane[f"p_short_{tag}"] = poisson.cdf(lane[clicks], lane[f"expected_{tag}"])
    lane[f"missed_{tag}"] = (lane[f"expected_{tag}"] - lane[clicks]).clip(lower=0)
    lane[f"ratio_{tag}"] = lane[clicks] / lane[f"expected_{tag}"]
lane["expected_A_tier"] = tier_expected(lane, "clicks_prev_30d", "impressions_prev_30d")
lane["ctr_A"] = 100 * lane["clicks_prev_30d"] / lane["impressions_prev_30d"]

show("Lane frame built", {
    "pages with position data and impressions in both windows": f"{len(lane):,}",
    "window A": "the earlier 30 days (used to score pages)",
    "window B": "the most recent 30 days (used only to judge the score)",
})

## 1. My lane as an ML task (type)

**Ranking / scoring, built in two stages.**

1. **Expected clicks:** an estimate of what a page _should_ earn from its impressions, given its position, its client, and later its content type and intent. Statistically this is a regression problem, and it is where a learned model can earn its place.
2. **The score:** missed clicks from real shortfalls, meaning expected minus actual clicks, kept only when luck would produce the gap less than 1 time in 100. Sorting pages by this score gives the queue.

Why not the other task types:

- **Not classification.** A yes/no "opportunity" label would have to be invented, and a model trained on an invented label just learns the rule back. A yes/no outcome does appear in Section 2, but only as something _observed_ a month later to judge the score, never as something the score is fitted to.
- **Not clustering.** Clusters say which kinds of pages exist (that is Lane 3). They cannot say which page in a cluster to open first.

**Method:** this cell shows both stages on six real pages: the three highest-scoring pages, then three zero-click pages that a CTR-gap rule would flag. It explains why a page with _zero_ clicks can rank _below_ pages that earned clicks.


In [ ]:
# The three top-scoring pages, then three zero-click pages a CTR-gap rule would flag.
lane["score_A"] = lane["missed_A"] * (lane["p_short_A"] < 0.01)
lane["tier_median_A"] = lane.groupby("position_tier")["ctr_A"].transform("median")
gap_flag = (lane["ctr_A"] < 0.5 * lane["tier_median_A"]) & (lane["impressions_prev_30d"] >= 100)

examples = pd.concat([
    lane.sort_values("score_A", ascending=False).head(3),
    lane[gap_flag & (lane["clicks_prev_30d"] == 0) & (lane["p_short_A"] > 0.05)]
    .sort_values("impressions_prev_30d", ascending=False).head(3),
])
example_table = pd.DataFrame({
    "Tier": examples["position_tier"],
    "Impressions (A)": examples["impressions_prev_30d"].map("{:,}".format),
    "Clicks (A)": examples["clicks_prev_30d"].map("{:,}".format),
    "Expected clicks": examples["expected_A"].map("{:,.1f}".format),
    "Missed clicks": examples["missed_A"].map("{:,.1f}".format),
    "p (luck)": examples["p_short_A"].map("{:.4f}".format),
    "Score": examples["score_A"].map("{:,.1f}".format),
}).set_index(examples["content_id"])
example_table.index.name = "Page (pseudonym)"
show_table("Top three pages by score, then three zero-click pages a CTR-gap rule would flag", example_table)

## 2. Target or proxy

Three different things get called "the target" in this lane, and keeping them apart is the core of the framing.

|                                   | Built from                                    | Role                             |
| --------------------------------- | --------------------------------------------- | -------------------------------- |
| **Score**                         | Window A clicks and impressions only          | What the queue is sorted by      |
| **Observed target**               | Window B, which the score never sees          | What the queue is judged against |
| **Self-defined label** (the trap) | The same window and the same gap as the score | Makes any ranking look good      |

**The observed target, `persistent_shortfall`:** in the most recent 30 days, the page earned clearly fewer clicks than its client-aware expectation (p below 0.05) _and_ at most half of what was expected. Window B's clicks and impressions are Search Console measurements, so this is an outcome the data recorded, not a rule I wrote. The "at most half" condition stops very large pages from qualifying on a gap that is statistically clear but too small to matter.

**Why this is not circular:** the score only uses window A. If a window-A shortfall were just luck, it would usually not show up again in window B.

**Limitations:**

- **Position overlaps the target window.** Position is a 90-day average covering both windows, so a little information about window B leaks into the expectation. The full warehouse records position daily, which would remove this overlap.
- **Repeating is not the same as fixable.** A page can fall short every month because of Google's AI answers or rich results that no rewrite can change. The editor's judgement is still required.
- **The window-B client factor uses window-B data.** That is fine for _defining_ the outcome, but it is not known in advance, so it never enters the score.

**Method:** this cell builds the target and reports its base rate, then demonstrates the trap by judging the same CTR-gap ranking two ways: against a label made from its own gap, and against the observed window-B target.


In [ ]:
# The target: a clear and substantial shortfall again in window B, which the score never sees.
lane["persistent_shortfall"] = ((lane["p_short_B"] < 0.05) & (lane["ratio_B"] <= 0.5)).astype(int)
target = lane["persistent_shortfall"]
base_rate = target.mean()
show("The observed target", {
    "pages with a persistent shortfall": f"{target.sum():,} of {len(lane):,}",
    "base rate (chance of a random pick being right)": f"{base_rate:.1%}",
})

# The trap: the same ranking judged against its own self-defined label, then against the real outcome.
gap_score = lane["tier_median_A"] - lane["ctr_A"]
self_label = gap_flag.astype(int)
show("CTR-gap ranking, judged two ways (Precision@50)", {
    "against its own self-defined label": f"{precision_at_k(gap_score, self_label, 50):.2f}",
    "against the observed window-B target": f"{precision_at_k(gap_score, target, 50):.2f}",
    "base rate, for comparison": f"{base_rate:.3f}",
})

## 3. Success metric

**Primary: Precision@50 against `persistent_shortfall`**, always printed beside the base rate. Of the 50 pages an editor would open, how many showed a real and substantial shortfall the following month?

**Secondary:**

- **Precision@20**, the view for a smaller review cycle.
- **Capture@50**, the share of all window-B missed clicks held by the top 50 pages. Precision counts pages; capture counts the clicks those pages represent, which is what the business actually loses.

**Locked before any model is trained:**

- A method only counts as useful if its Precision@50 is at least **3 times the base rate**.
- A learned model replaces the transparent baseline only if it beats the baseline's Precision@50 **without lowering Capture@50**, on a client holdout. Confidence intervals will then decide whether a gap between methods is real or noise.

**Method:** this cell scores four simple ranking methods on the same pages and the same target, so the bar any later model must clear is on the record before modelling starts. The last row is this project's transparent baseline.


In [ ]:
# Four transparent ranking methods, all scored on window A and judged on window B.
value = lane["missed_B"]
methods = {
    "Raw CTR, lowest first": -lane["ctr_A"],
    "CTR gap vs tier median": gap_score,
    "Missed clicks, tier expectation": (lane["expected_A_tier"] - lane["clicks_prev_30d"]).clip(lower=0),
    "Missed clicks from real shortfalls (baseline)": lane["score_A"],
}
results = pd.DataFrame([
    {
        "Method": name,
        "Precision@20": f"{precision_at_k(s, target, 20):.2f}",
        "Precision@50": f"{precision_at_k(s, target, 50):.2f}",
        "Lift @50 (x random)": f"{precision_at_k(s, target, 50) / base_rate:.1f}",
        "Capture@50": f"{capture_at_k(s, value, 50):.1%}",
    }
    for name, s in methods.items()
]).set_index("Method")
show_table("Four ranking methods compared on the observed target", results)
show("The bar a learned model must clear", {
    "base rate": f"{base_rate:.3f}",
    "minimum useful Precision@50 (3x base rate)": f"{3 * base_rate:.3f}",
    "a model must beat": "the baseline row, on a client holdout, without lowering Capture@50",
})

## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), carrying 90-day totals plus two separate 30-day windows.

**Method:** this cell shows how the 30,000 starter rows narrow down to the lane's pages, labels every column by its role (and why), checks that each page appears exactly once, and displays the eight highest-scoring pages with their window-B outcome.


In [ ]:
# How the starter rows narrow to the lane's pages.
funnel = pd.DataFrame({
    "Pages": [f"{len(df):,}", f"{(df['avg_position'] > 0).sum():,}", f"{len(lane):,}"],
}, index=pd.Index(["All starter rows", "With real position data", "With impressions in both 30-day windows"], name="Step"))
show_table("How the starter rows narrow to the lane's pages", funnel)

# What each column is for, and why.
roles = pd.DataFrame([
    ("content_id, client_id", "Grouping and splitting only", "Pseudonyms; never a model input"),
    ("position_tier, avg_position", "Input to expected clicks", "90-day average, so it overlaps window B"),
    ("impressions_prev_30d, clicks_prev_30d", "Score input (window A)", "The only clicks the score may see"),
    ("impressions_last_30d, clicks_last_30d", "Target only (window B)", "Never enters the score"),
    ("sessions_90d, engagement_rate", "Reason code", "Changes the action; only where analytics data exists"),
    ("trend_direction, trend_pct", "Excluded", "The starter's decline label is built from these"),
], columns=["Columns", "Role", "Why"]).set_index("Columns")
show_table("What each column is used for", roles)

assert lane["content_id"].is_unique, "Grain broken: a page appears more than once."
show("Grain check", {"each row is one unique page": "passed"})

top8 = lane.sort_values("score_A", ascending=False).head(8)
show_table("The eight highest-scoring pages and what happened next", pd.DataFrame({
    "Tier": top8["position_tier"],
    "Impressions (A)": top8["impressions_prev_30d"].map("{:,}".format),
    "Clicks (A)": top8["clicks_prev_30d"].map("{:,}".format),
    "Expected (A)": top8["expected_A"].map("{:,.0f}".format),
    "Score": top8["score_A"].map("{:,.0f}".format),
    "Clicks (B)": top8["clicks_last_30d"].map("{:,}".format),
    "Persistent shortfall (B)": top8["persistent_shortfall"].map({1: "yes", 0: "no"}),
}).set_index(top8["content_id"]).rename_axis("Page (pseudonym)"))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.